## BPE from schratch


In [5]:
from collections import Counter, deque


class BPETokenizerSimple:
    def __init__(self):
        
        
        # VOCABULARIO PRINCIPAL (ID → Token)
        # Mapea cada ID numérico a su representación de texto
        # Ejemplo: {0: 'a', 1: 'b', 2: 'c', 257: 'th', 258: 'ing'}
        self.vocab = {}
        
        # VOCABULARIO INVERSO (Token → ID)  
        # Mapea cada token de texto a su ID numérico único
        # Ejemplo: {'a': 0, 'b': 1, 'c': 2, 'th': 257, 'ing': 258}
        self.inverse_vocab = {}
        
        # REGLAS DE FUSIÓN BPE
        # Diccionario que guarda qué pares de tokens se fusionaron
        # Clave: tupla (token_id1, token_id2) - par original
        # Valor: nuevo_token_id - resultado de la fusión
        # Ejemplo: {(116, 104): 257} significa que 't' + 'h' se convierte en 'th'
        self.bpe_merges = {}


## FUNCIÓN 1: ENCUENTRA EL PAR MÁS FRECUENTE
    
Esta es la función que hace que BPE sea "inteligente".Analiza toda la secuencia de tokens y cuenta cuántas veces aparece cada par de tokens adyacentes. 
Es como un detective que busca patrones repetidos en el texto.

Returns:
    tuple o None: El par (token1_id, token2_id) más frecuente, o None si no hay pares


### Ejemplo de funcionamiento:

token_ids = [1, 2, 1, 2, 3, 1, 2]  # representa "ababcab"

Pares encontrados:
- (1,2) aparece 3 veces ← MÁS FRECUENTE

- (2,1) aparece 1 vez  

- (2,3) aparece 1 vez

- (3,1) aparece 1 vez

    Retorna: (1, 2)  # El par más frecuente

In [6]:
@staticmethod
def find_freq_pair(token_ids, mode="most"):
    # PASO 1: CREAR TODOS LOS PARES ADYACENTES
    # zip(token_ids, token_ids[1:]) crea pares de tokens consecutivos
    # Ejemplo: [1,2,3,4] → [(1,2), (2,3), (3,4)]
    pairs = Counter(zip(token_ids, token_ids[1:]))
    
    # PASO 2: VERIFICAR SI HAY PARES DISPONIBLES  
    if not pairs:
        # Si la lista está vacía o tiene un solo elemento, no hay pares
        return None
    
    # PASO 3: ENCONTRAR EL PAR SEGÚN EL CRITERIO
    if mode == "most":
        # max() encuentra el par con mayor frecuencia
        # key=lambda x: x[1] usa el conteo (segundo elemento) para comparar
        # [0] toma solo la tupla del par, no el conteo
        return max(pairs.items(), key=lambda x: x[1])[0]
    elif mode == "least":
        # min() encuentra el par con menor frecuencia (rara vez usado)
        return min(pairs.items(), key=lambda x: x[1])[0]
    else:
        raise ValueError("Modo inválido. Usa 'most' o 'least'.")
BPETokenizerSimple.find_freq_pair = find_freq_pair


## FUNCIÓN 2: REEMPLAZA TODAS LAS OCURRENCIAS DE UN PAR

Una vez que sabemos cuál par fusionar, esta función hace el trabajo:

encuentra todas las veces que aparece ese par y lo reemplaza con un nuevo token único. Es como un editor de texto que hace "buscar y reemplazar" inteligente.
Returns:
    List[int]: Nueva lista con los pares reemplazados
        
### Ejemplo de funcionamiento:
- token_ids = [116, 104, 101, 116, 104]  # "these"
- pair_id = (116, 104)  # "th" 
- new_id = 257  # nuevo token para "th"
        
    Resultado: [257, 101, 257]  # "th-e-th

In [7]:
@staticmethod 
def replace_pair(token_ids, pair_id, new_id):   
    # PASO 1: USAR DEQUE PARA PROCESAMIENTO EFICIENTE
    # deque permite agregar/quitar elementos del inicio/final eficientemente
    dq = deque(token_ids)
    replaced = []  # Lista resultado
    
    # PASO 2: PROCESAR CADA TOKEN SECUENCIALMENTE
    while dq:  # Mientras queden tokens por procesar
        # Tomar el primer token
        current = dq.popleft()
        
        # PASO 3: VERIFICAR SI FORMA EL PAR A REEMPLAZAR
        if dq and (current, dq[0]) == pair_id:
            # ¡ENCONTRAMOS EL PAR!
            # - dq[0] es el siguiente token 
            # - (current, dq[0]) forma exactamente el par que buscamos
            
            # Agregar el nuevo token fusionado
            replaced.append(new_id)
            
            # Quitar el segundo token del par (ya fue fusionado)
            dq.popleft() 
            
        else:
            # NO es el par que buscamos, mantener el token original
            replaced.append(current)
    
    return replaced
BPETokenizerSimple.replace_pair = replace_pair

## Método de Entrenamiento

In [8]:
def train(self, text, vocab_size, allowed_special={"<|endoftext|>"}):
    

    # Preprocesamiento: Reemplaza espacios con "Ġ"
    # Nota que Ġ es una particularidad de la implementación BPE de GPT-2
    # Ej: "Hola mundo" puede tokenizarse como ["Hola", "Ġmundo"]
    processed_text = []
    for i, char in enumerate(text):
        if char == " " and i != 0:
            processed_text.append("Ġ")
        if char != " ":
            processed_text.append(char)
    processed_text = "".join(processed_text)

    # Inicializa vocabulario con caracteres únicos, incluyendo "Ġ" si está presente
    # Comienza con los primeros 256 caracteres ASCII
    unique_chars = [chr(i) for i in range(256)]
    unique_chars.extend(
        char for char in sorted(set(processed_text))
        if char not in unique_chars
    )
    if "Ġ" not in unique_chars:
        unique_chars.append("Ġ")

    self.vocab = {i: char for i, char in enumerate(unique_chars)}
    self.inverse_vocab = {char: i for i, char in self.vocab.items()}

    # Agrega tokens especiales permitidos
    if allowed_special:
        for token in allowed_special:
            if token not in self.inverse_vocab:
                new_id = len(self.vocab)
                self.vocab[new_id] = token
                self.inverse_vocab[token] = new_id

    # Tokeniza el texto procesado en IDs de tokens
    token_ids = [self.inverse_vocab[char] for char in processed_text]

    # Pasos 1-3 de BPE: Repetidamente encuentra y reemplaza pares frecuentes
    for new_id in range(len(self.vocab), vocab_size):
        pair_id = self.find_freq_pair(token_ids, mode="most")
        if pair_id is None:
            break
        token_ids = self.replace_pair(token_ids, pair_id, new_id)
        self.bpe_merges[pair_id] = new_id

    # Construye el vocabulario con tokens fusionados
    for (p0, p1), new_id in self.bpe_merges.items():
        merged_token = self.vocab[p0] + self.vocab[p1]
        self.vocab[new_id] = merged_token
        self.inverse_vocab[merged_token] = new_id

# Agregar método a la clase
BPETokenizerSimple.train = train

In [9]:
def tokenize_with_bpe(self, token):
    """
    🧩 FUNCIÓN DE TOKENIZACIÓN BPE PARA TOKENS INDIVIDUALES
    
    Esta función aplica las reglas BPE aprendidas durante el entrenamiento
    a un token individual que no está en el vocabulario. Es el corazón
    de la tokenización subpalabra de BPE.
    
    Es como un rompecabezas: toma una palabra desconocida y la desarma
    en piezas que sí conoce usando las fusiones aprendidas.
    
    Args:
        token (str): Token individual a tokenizar (ej: "tokenización")
        
    Returns:
        List[int]: Lista de IDs que representan el token descompuesto
    """
    
    # ================================================================
    # PASO 1: CONVERSIÓN A CARACTERES INDIVIDUALES
    # ================================================================
    
    # Convierte cada carácter del token a su ID numérico
    # Ejemplo: "hola" → ['h', 'o', 'l', 'a'] → [104, 111, 108, 97]
    token_ids = [self.inverse_vocab.get(char, None) for char in token]
    
    # VERIFICACIÓN DE SEGURIDAD: ¿Todos los caracteres están en vocabulario?
    if None in token_ids:
        # Encuentra qué caracteres faltan para dar error informativo
        missing_chars = [char for char, tid in zip(token, token_ids) if tid is None]
        raise ValueError(f"Caracteres no encontrados en vocabulario: {missing_chars}")

    # ================================================================
    # PASO 2: APLICACIÓN ITERATIVA DE REGLAS BPE
    # ================================================================
    
    # Bandera que controla si aún se pueden hacer fusiones
    can_merge = True
    
    # BUCLE PRINCIPAL: Continúa mientras se puedan hacer fusiones
    while can_merge and len(token_ids) > 1:
        can_merge = False  # Asume que no habrá fusiones esta iteración
        new_tokens = []    # Lista para almacenar tokens después de fusiones
        i = 0              # Índice para recorrer la lista de tokens
        
        # ============================================================
        # PASO 2.1: BUSCAR Y APLICAR FUSIONES EN ESTA ITERACIÓN
        # ============================================================
        
        # Recorre todos los pares adyacentes de tokens
        while i < len(token_ids) - 1:
            # Crea par con token actual y siguiente
            pair = (token_ids[i], token_ids[i + 1])
            
            # ¿Este par tiene una regla de fusión aprendida?
            if pair in self.bpe_merges:
                # ¡SÍ! Aplicar la fusión
                
                # Obtiene el ID del token fusionado
                merged_token_id = self.bpe_merges[pair]
                
                # Agrega el token fusionado en lugar de los dos originales
                new_tokens.append(merged_token_id)
                
                # Salta ambos tokens del par (ya fueron fusionados)
                i += 2
                
                # Marca que hubo al menos una fusión en esta iteración
                can_merge = True
                
            else:
                # NO hay regla para este par, mantener token original
                new_tokens.append(token_ids[i])
                i += 1
        
        # ============================================================
        # PASO 2.2: MANEJAR TOKEN FINAL SI QUEDÓ SIN PROCESAR
        # ============================================================
        
        # Si el último token no fue parte de ningún par, agregarlo
        if i < len(token_ids):
            new_tokens.append(token_ids[i])
        
        # ============================================================
        # PASO 2.3: PREPARAR PARA SIGUIENTE ITERACIÓN
        # ============================================================
        
        # La nueva lista se convierte en la lista a procesar
        token_ids = new_tokens
        
        # El bucle continuará si can_merge=True (hubo fusiones)
        # y si quedan al menos 2 tokens para formar pares
    
    # ================================================================
    # PASO 3: RETORNAR RESULTADO FINAL
    # ================================================================
    
    # Retorna la lista de IDs después de aplicar todas las fusiones posibles
    return token_ids

# Asigna la función a la clase
BPETokenizerSimple.tokenize_with_bpe = tokenize_with_bpe


In [10]:
def encode(self, text, allowed_special=None):
    import re  # Importa regex para manejo de tokens especiales
    
    token_ids = []  # Lista final de IDs que se retornará
    
    # ===============================================================
    # FASE 1: MANEJO DE TOKENS ESPECIALES (SI ESTÁ HABILITADO)
    # ===============================================================
    if allowed_special is not None and len(allowed_special) > 0:
        
        # Construye patrón regex para encontrar tokens especiales
        # Los ordena por longitud (más largo primero) para evitar matches parciales
        # Ejemplo: si tienes "<|end|>" y "<|endoftext|>", busca primero el más largo
        special_pattern = (
            "(" + "|".join(re.escape(tok) for tok in sorted(allowed_special, key=len, reverse=True)) + ")"
        )
        
        last_index = 0  # Rastrea posición en el texto
        
        # Busca cada ocurrencia de tokens especiales en el texto
        for match in re.finditer(special_pattern, text):
            # PROCESA TEXTO ANTES DEL TOKEN ESPECIAL
            prefix = text[last_index:match.start()]
            # Codifica recursivamente el prefijo SIN manejo de tokens especiales
            token_ids.extend(self.encode(prefix, allowed_special=None))
            
            # PROCESA EL TOKEN ESPECIAL ENCONTRADO
            special_token = match.group(0)
            if special_token in self.inverse_vocab:
                # Agrega directamente el ID del token especial
                token_ids.append(self.inverse_vocab[special_token])
            else:
                # Error si el token especial no está en vocabulario
                raise ValueError(f"Token especial {special_token} no encontrado en vocabulario.")
            
            # Actualiza posición para continuar después del token especial
            last_index = match.end()
        
        # Obtiene texto restante después del último token especial
        text = text[last_index:]
        
        # VERIFICACIÓN DE SEGURIDAD: Busca tokens especiales no permitidos
        # Encuentra tokens con formato <|...| > que no están en allowed_special
        disallowed = [
            tok for tok in self.inverse_vocab
            if tok.startswith("<|") and tok.endswith("|>") and tok in text and tok not in allowed_special
        ]
        if disallowed:
            raise ValueError(f"Tokens especiales no permitidos encontrados en texto: {disallowed}")
    
    # ===============================================================
    # FASE 2: TOKENIZACIÓN PRINCIPAL DEL TEXTO
    # ===============================================================
    
    tokens = []  # Lista temporal de tokens de texto
    
    # MANEJO DE MÚLTIPLES LÍNEAS
    lines = text.split("\n")  # Divide texto en líneas
    
    for i, line in enumerate(lines):
        # Agrega símbolo de nueva línea para líneas después de la primera
        if i > 0:
            tokens.append("\n")
        
        # MANEJO DE PALABRAS DENTRO DE CADA LÍNEA
        words = line.split()  # Divide línea en palabras (separadas por espacios)
        
        for j, word in enumerate(words):
            # LÓGICA DE ESPACIOS CON SÍMBOLO "Ġ":
            # - Primera palabra después de nueva línea: lleva "Ġ" (representa espacio)
            # - Primera palabra de primera línea: sin "Ġ"
            # - Resto de palabras: llevan "Ġ" (representan espacios anteriores)
            
            if j == 0 and i > 0:
                # Primera palabra después de nueva línea
                tokens.append("Ġ" + word)
            elif j == 0:
                # Primera palabra de la primera línea
                tokens.append(word)
            else:
                # Palabras subsecuentes (llevan espacio representado por "Ġ")
                tokens.append("Ġ" + word)
    
    # ===============================================================
    # FASE 3: CONVERSIÓN DE TOKENS A IDs
    # ===============================================================
    
    for token in tokens:
        if token in self.inverse_vocab:
            # Token conocido: busca directamente su ID
            token_ids.append(self.inverse_vocab[token])
        else:
            # Token desconocido: aplica algoritmo BPE para descomponerlo
            # Esta llamada usa las reglas de fusión aprendidas durante entrenamiento
            token_ids.extend(self.tokenize_with_bpe(token))
    
    return token_ids  # Retorna lista final de IDs de tokens

# Asigna la función a la clase
BPETokenizerSimple.encode = encode



In [11]:
def decode(self, token_ids):

    
    # ================================================================
    # INICIALIZACIÓN
    # ================================================================
    
    # String que acumulará el texto final decodificado
    decoded_string = ""
    
    # ================================================================
    # BUCLE PRINCIPAL: PROCESA CADA TOKEN ID
    # ================================================================
    
    # Itera sobre cada ID de token con su índice
    for i, token_id in enumerate(token_ids):
        
        # ============================================================
        # PASO 1: VERIFICACIÓN DE SEGURIDAD
        # ============================================================
        
        # ¿Este ID existe en nuestro vocabulario?
        if token_id not in self.vocab:
            raise ValueError(f"Token ID {token_id} no encontrado en vocabulario.")
        
        # Obtiene el string correspondiente al ID
        token = self.vocab[token_id]  # Ej: 123 → "Hola"
        
        # ============================================================
        # PASO 2: MANEJO ESPECIAL DE SALTOS DE LÍNEA
        # ============================================================
        
        if token == "\n":
            # LÓGICA ESPECIAL PARA NUEVA LÍNEA:
            # Si no hay espacio antes del salto de línea, agregarlo
            if decoded_string and not decoded_string.endswith(" "):
                decoded_string += " "  # Agrega espacio para mejor formato
            
            # Agrega el salto de línea
            decoded_string += token
            
        # ============================================================
        # PASO 3: MANEJO DEL SÍMBOLO ESPECIAL "Ġ" (ESPACIOS)
        # ============================================================
        
        elif token.startswith("Ġ"):
            # TOKENS QUE REPRESENTAN ESPACIOS:
            # "Ġ" es el símbolo especial que representa espacios iniciales
            # Ejemplo: "Ġmundo" significa " mundo" (con espacio al inicio)
            
            # Convierte "Ġ" de vuelta a espacio real y agrega el resto del token
            decoded_string += " " + token[1:]  # "Ġmundo" → " mundo"
            
        # ============================================================
        # PASO 4: TOKENS NORMALES (SIN ESPACIOS ESPECIALES)
        # ============================================================
        
        else:
            # TOKENS REGULARES:
            # No tienen símbolos especiales, se agregan directamente
            # Ejemplo: "Hola", "123", "!", etc.
            decoded_string += token
    
    # ================================================================
    # PASO 5: RETORNAR RESULTADO FINAL
    # ================================================================
    
    # Retorna el string completamente reconstruido
    return decoded_string

# Asigna la función a la clase
BPETokenizerSimple.decode = decode


## Entrenamiento corpus Colombiano

In [12]:
# Cargar el corpus colombiano desde archivo
with open('corpus_colombia.txt', 'r', encoding='utf-8') as f:
    corpus_colombiano = f.read()

print(f"   Caracteres: {len(corpus_colombiano):,}")
print(f"   Palabras aproximadas: {len(corpus_colombiano.split()):,}")

   Caracteres: 10,019
   Palabras aproximadas: 1,566


In [13]:

print("\n🚀 Creando tokenizador BPE para texto colombiano...")
tokenizer_colombiano = BPETokenizerSimple()

# Verificar que está inicializado correctamente
print(f"   Vocabulario inicial: {len(tokenizer_colombiano.vocab)} tokens")
print(f"   Fusiones iniciales: {len(tokenizer_colombiano.bpe_merges)} fusiones")


🚀 Creando tokenizador BPE para texto colombiano...
   Vocabulario inicial: 0 tokens
   Fusiones iniciales: 0 fusiones


In [14]:
# Parámetros de entrenamiento
vocab_size = 1000  # Tamaño del vocabulario final
tokens_especiales = {"<|endoftext|>"}  # Tokens especiales a incluir

print(f"\n⚙️ Configuración de entrenamiento:")
print(f"   Tamaño de vocabulario objetivo: {vocab_size}")
print(f"   Tokens especiales: {tokens_especiales}")
print(f"   Fusiones a aprender: {vocab_size - 256 - len(tokens_especiales) - 1} fusiones")
# 256 caracteres base + tokens especiales + símbolo Ġ



⚙️ Configuración de entrenamiento:
   Tamaño de vocabulario objetivo: 1000
   Tokens especiales: {'<|endoftext|>'}
   Fusiones a aprender: 742 fusiones


In [15]:
import time

print(f"\n🏋️ Iniciando entrenamiento BPE...")

# Medir tiempo de entrenamiento
start_time = time.time()

# ¡ENTRENAR!
tokenizer_colombiano.train(
    text=corpus_colombiano,
    vocab_size=vocab_size,
    allowed_special=tokens_especiales
)

end_time = time.time()
training_time = end_time - start_time

print(f"\n✅ ¡Entrenamiento completado!")
print(f"   Tiempo transcurrido: {training_time:.2f} segundos")
print(f"   Vocabulario final: {len(tokenizer_colombiano.vocab)} tokens")
print(f"   Fusiones aprendidas: {len(tokenizer_colombiano.bpe_merges)} fusiones")



🏋️ Iniciando entrenamiento BPE...

✅ ¡Entrenamiento completado!
   Tiempo transcurrido: 0.60 segundos
   Vocabulario final: 1000 tokens
   Fusiones aprendidas: 742 fusiones


In [17]:
# Ejemplos de prueba con texto colombiano
ejemplos_test = [
    "Gabriel García Márquez",
    "Bogotá Colombia",
    "café colombiano",
    "vallenato y bambuco",
    "Universidad Nacional"
]

print(f"\n🧪 Pruebas de tokenización:")
for i, ejemplo in enumerate(ejemplos_test, 1):
    # Codificar
    tokens = tokenizer_colombiano.encode(ejemplo)
    tokens_str = [tokenizer_colombiano.vocab[t] for t in tokens]
    
    # Decodificar para verificar
    decodificado = tokenizer_colombiano.decode(tokens)
    correcto = ejemplo == decodificado
    
    print(f"\n   Prueba {i}: '{ejemplo}'")
    print(f"   Tokens ({len(tokens)}): {tokens_str}")
    print(f"   IDs: {tokens}")
    print(f"   Decodificado: '{decodificado}'")
    print(f"   ✅ Correcto: {correcto}")



🧪 Pruebas de tokenización:

   Prueba 1: 'Gabriel García Márquez'
   Tokens (16): ['Ga', 'br', 'i', 'el', 'Ġ', 'Ga', 'r', 'c', 'í', 'a', 'Ġ', 'M', 'á', 'r', 'qu', 'ez']
   IDs: [601, 353, 105, 310, 256, 601, 114, 99, 237, 97, 256, 77, 225, 114, 295, 667]
   Decodificado: 'Gabriel García Márquez'
   ✅ Correcto: True

   Prueba 2: 'Bogotá Colombia'
   Tokens (4): ['Bogotá', 'Ġ', 'Colombi', 'a']
   IDs: [359, 256, 317, 97]
   Decodificado: 'Bogotá Colombia'
   ✅ Correcto: True

   Prueba 3: 'café colombiano'
   Tokens (7): ['café', 'Ġ', 'col', 'om', 'bi', 'an', 'o']
   IDs: [773, 256, 521, 281, 276, 267, 111]
   Decodificado: 'café colombiano'
   ✅ Correcto: True

   Prueba 4: 'vallenato y bambuco'
   Tokens (12): ['v', 'al', 'le', 'n', 'at', 'o', 'Ġ', 'y', 'Ġ', 'bamb', 'uc', 'o']
   IDs: [118, 274, 994, 110, 348, 111, 256, 121, 256, 641, 818, 111]
   Decodificado: 'vallenato y bambuco'
   ✅ Correcto: True

   Prueba 5: 'Universidad Nacional'
   Tokens (7): ['Universi', 'dad', 'Ġ', 'N', 

In [ ]:
# Prueba final con un texto más largo
texto_prueba = """
    Los tombos 

"""

print(f"\n🎯 Prueba final completa:")
print(f"Texto original ({len(texto_prueba)} chars):")
print(f"'{texto_prueba.strip()}'")

# Tokenizar
tokens_finales = tokenizer_colombiano.encode(texto_prueba)
tokens_str_finales = [tokenizer_colombiano.vocab[t] for t in tokens_finales]

print(f"\nTokenización ({len(tokens_finales)} tokens):")
print(f"Tokens: {tokens_str_finales}")

# Calcular ratio de compresión
ratio_compresion = len(texto_prueba) / len(tokens_finales)
print(f"\n📈 Ratio de compresión: {ratio_compresion:.2f} (chars/token)")

# Verificar reversibilidad
decodificado_final = tokenizer_colombiano.decode(tokens_finales)
print(f"\n🔄 Texto decodificado:")
print(f"'{decodificado_final.strip()}'")
print(f"✅ Reversibilidad correcta: {texto_prueba.strip() == decodificado_final.strip()}")




🎯 Prueba final completa:
Texto original (18 chars):
'Los tombos'

Tokenización (11 tokens):
Tokens: ['\n', 'Ġ', 'L', 'os', 'Ġ', 'to', 'm', 'bo', 's', '\n', '\n']

📈 Ratio de compresión: 1.64 (chars/token)

🔄 Texto decodificado:
'Los tombos'
✅ Reversibilidad correcta: True

🎉 ¡Entrenamiento del tokenizador colombiano completado exitosamente!


## Corpus de codigo

In [20]:
# Cargar el corpus de código desde archivo
with open('corpus_codigo.txt', 'r', encoding='utf-8') as f:
    corpus_codigo = f.read()

# Verificar que se cargó correctamente
print("💻 Corpus de Código Cargado:")
print(f"   Caracteres: {len(corpus_codigo):,}")
print(f"   Líneas de código: {len(corpus_codigo.splitlines()):,}")
print(f"   Palabras/tokens aprox: {len(corpus_codigo.split()):,}")

💻 Corpus de Código Cargado:
   Caracteres: 2,912
   Líneas de código: 87
   Palabras/tokens aprox: 238


In [21]:
# Crear nueva instancia del tokenizador BPE para código
print("\n🚀 Creando tokenizador BPE para código Python...")
tokenizer_codigo = BPETokenizerSimple()

# Verificar inicialización
print(f"   Vocabulario inicial: {len(tokenizer_codigo.vocab)} tokens")
print(f"   Fusiones iniciales: {len(tokenizer_codigo.bpe_merges)} fusiones")
print("   Estado: Listo para entrenamiento")


🚀 Creando tokenizador BPE para código Python...
   Vocabulario inicial: 0 tokens
   Fusiones iniciales: 0 fusiones
   Estado: Listo para entrenamiento


In [22]:
# Parámetros de entrenamiento para código
vocab_size_codigo = 1000  # Mismo tamaño que el tokenizador de texto para comparar
tokens_especiales_codigo = {"<|endoftext|>"}  # Tokens especiales

print(f"\n⚙️ Configuración de entrenamiento para código:")
print(f"   Tamaño de vocabulario objetivo: {vocab_size_codigo}")
print(f"   Tokens especiales: {tokens_especiales_codigo}")
print(f"   Fusiones a aprender: {vocab_size_codigo - 256 - len(tokens_especiales_codigo) - 1}")
print(f"   Diferencia con tokenizador texto: Especializándose en sintaxis de programación")


⚙️ Configuración de entrenamiento para código:
   Tamaño de vocabulario objetivo: 1000
   Tokens especiales: {'<|endoftext|>'}
   Fusiones a aprender: 742
   Diferencia con tokenizador texto: Especializándose en sintaxis de programación


In [ ]:
import time

print(f"\n🏋️ Iniciando entrenamiento BPE para código Python...")

# Medir tiempo de entrenamiento
start_time = time.time()

# ¡ENTRENAR PARA CÓDIGO!
tokenizer_codigo.train(
    text=corpus_codigo,
    vocab_size=vocab_size_codigo,
    allowed_special=tokens_especiales_codigo
)

end_time = time.time()
training_time = end_time - start_time

print(f"\n✅ ¡Entrenamiento de código completado!")
print(f"   Tiempo transcurrido: {training_time:.2f} segundos")
print(f"   Vocabulario final: {len(tokenizer_codigo.vocab)} tokens")
print(f"   Fusiones aprendidas: {len(tokenizer_codigo.bpe_merges)} fusiones")



🏋️ Iniciando entrenamiento BPE para código Python...
   Corpus: código Python (funciones, clases, imports)
   Esperamos que aprenda patrones como: 'def', 'import', '.py', '()', etc.

✅ ¡Entrenamiento de código completado!
   Tiempo transcurrido: 0.11 segundos
   Vocabulario final: 1000 tokens
   Fusiones aprendidas: 742 fusiones


In [ ]:
# Ejemplos de prueba con código típico
ejemplos_codigo_test = [
    "def calculate_fibonacci(n):",
    "import tensorflow as tf",
    "from sklearn import model_selection",
    "class NeuralNetwork:",
    "return np.dot(X_train, weights)",
    "for epoch in range(epochs):",
    "tokenizer = AutoTokenizer.from_pretrained"
]

print(f"\n🧪 Pruebas de tokenización:")
for i, ejemplo in enumerate(ejemplos_codigo_test, 1):
    # Codificar
    tokens = tokenizer_codigo.encode(ejemplo)
    tokens_str = [tokenizer_codigo.vocab[t] for t in tokens]
    
    # Decodificar para verificar
    decodificado = tokenizer_codigo.decode(tokens)
    correcto = ejemplo == decodificado
    
    print(f"\n   Prueba {i}: '{ejemplo}'")
    print(f"   Tokens ({len(tokens)}): {tokens_str}")
    print(f"   IDs: {tokens}")
    print(f"   Decodificado: '{decodificado}'")
    print(f"   ✅ Correcto: {correcto}")




🧪 Pruebas de tokenización:

   Prueba 1: 'def calculate_fibonacci(n):'
   Tokens (5): ['de', 'f', 'Ġ', 'calculate_fibonacci(n', '):']
   IDs: [276, 102, 256, 428, 288]
   Decodificado: 'def calculate_fibonacci(n):'
   ✅ Correcto: True

   Prueba 2: 'import tensorflow as tf'
   Tokens (11): ['import', 'Ġ', 'tensor', 'f', 'lo', 'w', 'Ġ', 'as', 'Ġ', 't', 'f']
   IDs: [456, 256, 564, 102, 289, 119, 256, 295, 256, 116, 102]
   Decodificado: 'import tensorflow as tf'
   ✅ Correcto: True

   Prueba 3: 'from sklearn import model_selection'
   Tokens (18): ['from', 'Ġ', 's', 'k', 'le', 'ar', 'n', 'Ġ', 'im', 'po', 'r', 't', 'Ġ', 'model', '_s', 'e', 'le', 'ction']
   IDs: [459, 256, 115, 107, 317, 340, 110, 256, 453, 296, 114, 116, 256, 460, 278, 101, 317, 357]
   Decodificado: 'from sklearn import model_selection'
   ✅ Correcto: True

   Prueba 4: 'class NeuralNetwork:'
   Tokens (7): ['class', 'Ġ', 'Ne', 'ural', 'Ne', 'twork', ':']
   IDs: [480, 256, 590, 511, 590, 514, 58]
   Decodificado: 'c

In [ ]:
codigo_complejo = """
def train_deep_neural_network(X_train, y_train, layers=[128, 64, 32], dropout=0.2):
    model = tf.keras.Sequential([
        tf.keras.layers.Dense(layers[0], activation='relu', input_shape=(X_train.shape[1],)),
        tf.keras.layers.Dropout(dropout),
        tf.keras.layers.Dense(layers[1], activation='relu'),
        tf.keras.layers.Dense(layers[2], activation='relu'),
        tf.keras.layers.Dense(1, activation='sigmoid')
    ])
"""

print("CÓDIGO ORIGINAL:")
print(codigo_complejo)

print("\n" + "="*60)

# Codificar
tokens_complejo = tokenizer_codigo.encode(codigo_complejo)

# Decodificar
decodificado_complejo = tokenizer_codigo.decode(tokens_complejo)

print("CÓDIGO DECODIFICADO:")
print(decodificado_complejo)

print("\n" + "="*60)
print(f"¿Son idénticos? {codigo_complejo == decodificado_complejo}")


CÓDIGO ORIGINAL:

def train_deep_neural_network(X_train, y_train, layers=[128, 64, 32], dropout=0.2):
    model = tf.keras.Sequential([
        tf.keras.layers.Dense(layers[0], activation='relu', input_shape=(X_train.shape[1],)),
        tf.keras.layers.Dropout(dropout),
        tf.keras.layers.Dense(layers[1], activation='relu'),
        tf.keras.layers.Dense(layers[2], activation='relu'),
        tf.keras.layers.Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model


CÓDIGO DECODIFICADO:

 def train_deep_neural_network(X_train, y_train, layers=[128, 64, 32], dropout=0.2): 
 model = tf.keras.Sequential([ 
 tf.keras.layers.Dense(layers[0], activation='relu', input_shape=(X_train.shape[1],)), 
 tf.keras.layers.Dropout(dropout), 
 tf.keras.layers.Dense(layers[1], activation='relu'), 
 tf.keras.layers.Dense(layers[2], activation='relu'), 
 tf.keras.layers.Dense(1, activation='sigmoid') 
 ]) 
 model.comp